<a href="https://colab.research.google.com/github/sheicksen/CISC483-EngageTactics/blob/JiaQi-RNN-Implementation/Truthseeker_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Attempted RNN Implementation

In [1]:
import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from google.colab import files

In [2]:
uploaded = files.upload()

# Assuming you upload a file named 'your_file_name.csv'
# You will need to replace 'your_file_name.csv' with the actual name of your uploaded file.

Saving Truth_Seeker_Model_Dataset.csv to Truth_Seeker_Model_Dataset.csv


In [3]:
# Get the name of the uploaded file(s)
# This assumes only one file is uploaded. If multiple, you'll need to choose the correct one.
for fn in uploaded.keys():
  print(f'User uploaded file "{fn}"')
  truthseeker_df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))

# Display the first few rows of the DataFrame, shape and headers
#display(truthseeker_df.head())
print(truthseeker_df.shape)

# print(truthseeker_df['tweet'][0])
print(truthseeker_df.info()) # print df info

User uploaded file "Truth_Seeker_Model_Dataset.csv"
(134198, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134198 entries, 0 to 134197
Data columns (total 9 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Unnamed: 0               134198 non-null  int64  
 1   author                   134198 non-null  object 
 2   statement                134198 non-null  object 
 3   target                   134198 non-null  bool   
 4   BinaryNumTarget          134198 non-null  float64
 5   manual_keywords          134198 non-null  object 
 6   tweet                    134198 non-null  object 
 7   5_label_majority_answer  134198 non-null  object 
 8   3_label_majority_answer  134198 non-null  object 
dtypes: bool(1), float64(1), int64(1), object(6)
memory usage: 8.3+ MB
None


In [4]:
# Define target (y) and features (X)
y = truthseeker_df['target']

# Drop target, uneeded, and text columns for now
X = truthseeker_df['tweet']

# split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (107358,)
X_test shape: (26840,)
y_train shape: (107358,)
y_test shape: (26840,)


In [5]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Parameters for tokenization and padding
VOCAB_SIZE = 10000  # Max number of words to keep, based on word frequency
MAX_SEQUENCE_LENGTH = 100 # Max length of each tweet sequence

# Initialize tokenizer
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")

# Fit tokenizer on the tweet column
tokenizer.fit_on_texts(X)

# Convert text to sequences of numbers
training_sequences = tokenizer.texts_to_sequences(X_train)
testing_sequences = tokenizer.texts_to_sequences(X_test)

# Pad sequences to ensure uniform length
padded_training_sequences = pad_sequences(training_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
padded_testing_sequences = pad_sequences(testing_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

print(f"Shape of padded training sequences: {padded_training_sequences.shape}")
print(f"Shape of padded testing sequences: {padded_testing_sequences.shape}")
print(f"Example of a padded sequence:\n{padded_training_sequences[1]}")
print(f"Vocabulary size: {len(tokenizer.word_index)}")

Shape of padded training sequences: (107358, 100)
Shape of padded testing sequences: (26840, 100)
Example of a padded sequence:
[   1  653    4  239  349  228   55  676 1260   52  300  315 4102  121
    3   23 1849   20    1    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0]
Vocabulary size: 169377


This `tokenizer.json` file can later be loaded to reconstruct the exact same tokenizer, which is crucial for using the trained model with new, unseen text data consistently.

The `padded_sequences` now contain the numerical representation of your tweets, ready to be fed into an embedding layer. Each number corresponds to a word in the tokenizer's vocabulary. The next step would typically involve creating a Keras Embedding layer as part of an RNN model.

In [6]:
import json

# Save the tokenizer configuration to a JSON file
tokenizer_json = tokenizer.to_json()
with open('tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(json.dumps(tokenizer_json, ensure_ascii=False))

print("Tokenizer saved to tokenizer.json")

Tokenizer saved to tokenizer.json


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# Define the RNN model
embedding_dim = 128 # The size of the vector space in which words will be embedded

# Embedding: Take the words (int form) and represents them as a vector of numbers.
           # Training adjusts those numbers, to reflect how the word was used in training
           # data. Words with similar meanings would be used in similar context, and so would
           # have similar embeddings.
# units=64 defines dimension of output, chosen as a good default value
#
model = Sequential([
    Embedding(VOCAB_SIZE, embedding_dim, input_length=MAX_SEQUENCE_LENGTH),
    SimpleRNN(units=64),
    Dense(1, activation='sigmoid') # Binary classification output
])

# Compile the model, defines how it will train
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
from tensorflow.keras.callbacks import EarlyStopping

# Convert boolean target to integer (0 or 1)
y_train_int = y_train.astype(int)
y_test_int = y_test.astype(int)

# Define EarlyStopping callback
# It monitors 'val_loss' and stops if it doesn't improve for 3 consecutive epochs.
# 'restore_best_weights' ensures the model keeps the weights from the epoch with the best validation loss.
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Train the model
history = model.fit(
    padded_training_sequences,
    y_train_int,
    epochs=10, # Start with a reasonable number of epochs, EarlyStopping will stop it if needed
    batch_size=64,
    validation_split=0.2, # Use 20% of training data for validation
    callbacks=[early_stopping]
)

print("Model training complete.")

Epoch 1/10
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 76s 56ms/step - accuracy: 0.5059 - loss: 0.6937 - val_accuracy: 0.5152 - val_loss: 0.6935
Epoch 2/10
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 77s 58ms/step - accuracy: 0.5070 - loss: 0.6935 - val_accuracy: 0.4844 - val_loss: 0.6963
Epoch 3/10
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 76s 56ms/step - accuracy: 0.5062 - loss: 0.6935 - val_accuracy: 0.5091 - val_loss: 0.6996
Epoch 4/10
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 82s 56ms/step - accuracy: 0.5078 - loss: 0.6931 - val_accuracy: 0.5159 - val_loss: 0.6937
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.
Model training complete.


In [12]:
# Evaluate the model on the test data
loss, accuracy = model.evaluate(padded_testing_sequences, y_test_int)

print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

839/839 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.5126 - loss: 0.6938

Test Loss: 0.6938
Test Accuracy: 0.5126
